In [8]:
import os
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support, classification_report

def calculate_and_display_metrics():
    # ============================================================================
    # ===== CONFIGURATION =====
    # ============================================================================
    # Match these to your generating script
    RUN_NAME = "modern-0513-173" 
    MODEL_NAME = "ModernBERT"
    RUN_DATE = "0513"
    RUN_INDEX = 173
    CLASSIFICATION_MISSING_VALUE = -100
    
    # Attributes saved by your evaluation script
    attrs = ["scale", "negative", "tag", "time"] 
    result_dir = "result"
    
    print("\n" + "="*70)
    print("📊 MACRO AND WEIGHTED METRICS REPORT")
    print("="*70)
    
    summary_data = []

    for attr in attrs:
        file_path = os.path.join(result_dir, f"{MODEL_NAME}_{RUN_DATE}_{RUN_INDEX}_result_{attr}.csv")
        
        if not os.path.exists(file_path):
            print(f"⚠️ Warning: File not found for '{attr}' at {file_path}")
            continue
            
        # Load the generated CSV
        df = pd.read_csv(file_path)
        
        true_col = f"true_{attr}"
        pred_col = f"pred_{attr}"
        
        if true_col not in df.columns or pred_col not in df.columns:
            print(f"⚠️ Warning: Missing expected columns in {file_path}")
            continue
        
        # Clean data: Filter out the ignore index (-100) from ground truths
        valid_mask = df[true_col] != CLASSIFICATION_MISSING_VALUE
        y_true = df.loc[valid_mask, true_col]
        y_pred = df.loc[valid_mask, pred_col]
        
        if len(y_true) == 0:
            print(f"⚠️ Warning: No valid samples found for '{attr}' after filtering missing values.")
            continue

        # ============================================================================
        # ===== CALCULATE METRICS =====
        # ============================================================================
        # Macro Avg: Calculates metrics for each label, and finds their unweighted mean. 
        # Weighted Avg: Calculates metrics for each label, and finds their average weighted by support.
        macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
        weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)
        
        # Save to summary dictionary for an aggregated view later
        summary_data.append({
            "Task": attr.upper(),
            "Macro P": round(macro_p, 4),
            "Macro R": round(macro_r, 4),
            "Macro F1": round(macro_f1, 4),
            "Weighted P": round(weighted_p, 4),
            "Weighted R": round(weighted_r, 4),
            "Weighted F1": round(weighted_f1, 4),
            "Valid Samples": len(y_true)
        })

        # Print individual task breakdown
        print(f"\n📌 Task: {attr.upper()} (Samples: {len(y_true)})")
        print("-" * 55)
        print(f"{'Metric':<15} | {'Macro Avg':<15} | {'Weighted Avg':<15}")
        print("-" * 55)
        print(f"{'Precision':<15} | {macro_p:<15.4f} | {weighted_p:<15.4f}")
        print(f"{'Recall':<15} | {macro_r:<15.4f} | {weighted_r:<15.4f}")
        print(f"{'F1-Score':<15} | {macro_f1:<15.4f} | {weighted_f1:<15.4f}")
        print("-" * 55)

    # ============================================================================
    # ===== AGGREGATED DATAFRAME =====
    # ============================================================================
    if summary_data:
        summary_df = pd.DataFrame(summary_data)
        print("\n" + "="*70)
        print("📈 CONSOLIDATED METRICS SUMMARY")
        print("="*70)
        print(summary_df.to_string(index=False))
        
        # Optional: Save the aggregated metrics to a new CSV
        summary_path = os.path.join(result_dir, f"{MODEL_NAME}_{RUN_DATE}_{RUN_INDEX}_aggregated_metrics.csv")
        summary_df.to_csv(summary_path, index=False)
        print(f"\n✅ Saved aggregated metrics to {summary_path}")

if __name__ == "__main__":
    calculate_and_display_metrics()


📊 MACRO AND WEIGHTED METRICS REPORT

📌 Task: SCALE (Samples: 82580)
-------------------------------------------------------
Metric          | Macro Avg       | Weighted Avg   
-------------------------------------------------------
Precision       | 0.3792          | 0.9812         
Recall          | 0.3896          | 0.9813         
F1-Score        | 0.3841          | 0.9812         
-------------------------------------------------------

📌 Task: NEGATIVE (Samples: 89383)
-------------------------------------------------------
Metric          | Macro Avg       | Weighted Avg   
-------------------------------------------------------
Precision       | 0.9011          | 0.9805         
Recall          | 0.7424          | 0.9824         
F1-Score        | 0.8010          | 0.9802         
-------------------------------------------------------

📌 Task: TAG (Samples: 90334)
-------------------------------------------------------
Metric          | Macro Avg       | Weighted Avg   
------